# LFW Grad-CAM — 03. Saliency feature analysis

그림을 선택적으로 보여주기 전에 모든 고정 사례에서 entropy와 중앙
영역 saliency 집중도를 동일한 정의로 계산합니다. 이 기술 통계만으로
압축 오류의 원인을 단정하지 않습니다.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(D:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_NAME = "arcface"     # "arcface", "adaface", "magface" 중 이번 실행 모델
MODE = "dev"               # 빠른 검증은 "dev", 전체 논문 실행만 "real"
DATA_FRACTION = 0.10       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합·tie-break·random control 재현 seed
EXECUTE_STAGE = False      # 입력과 checkpoint를 채운 뒤 실제 계산할 때만 True
WRITE_OUTPUTS = False      # 검증 후 새 artifact를 저장할 때만 True

if MODEL_NAME not in CONFIG["models"]["selected"]:
    raise ValueError(f"지원하지 않는 모델: {MODEL_NAME}")
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

In [ ]:
import numpy as np
import pandas as pd

from research.explainability.gradcam import (
    central_region_concentration,
    saliency_entropy,
)

CASE_MANIFEST_PATH = None
HEATMAP_INPUT_PATH = None
SALIENCY_FEATURES_PATH = None

In [ ]:
if EXECUTE_STAGE:
    if CASE_MANIFEST_PATH is None or HEATMAP_INPUT_PATH is None:
        raise RuntimeError("case manifest와 heatmap 경로를 지정하세요.")
    cases = pd.read_parquet(CASE_MANIFEST_PATH)
    heatmap_bundle = np.load(HEATMAP_INPUT_PATH, allow_pickle=False)
    case_ids = heatmap_bundle["case_id"].astype(str)
    if not np.array_equal(case_ids, cases["case_id"].astype(str).to_numpy()):
        raise ValueError("heatmap과 case manifest의 case_id 순서가 다릅니다.")
    heatmaps = heatmap_bundle["heatmaps"]
    features = cases.copy()
    features["origin_pair_score"] = heatmap_bundle["target_scores"]
    features["saliency_entropy"] = saliency_entropy(heatmaps)
    features["central_concentration_50pct"] = central_region_concentration(
        heatmaps,
        height_fraction=0.5,
        width_fraction=0.5,
    )
    if WRITE_OUTPUTS:
        if SALIENCY_FEATURES_PATH is None:
            raise RuntimeError("SALIENCY_FEATURES_PATH를 지정하세요.")
        destination = Path(SALIENCY_FEATURES_PATH).resolve()
        if destination.exists():
            raise FileExistsError(f"기존 feature를 덮어쓸 수 없습니다: {destination}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        features.to_parquet(destination, index=False)
    feature_summary = features.groupby(
        ["compression_family", "compression_profile", "case_group"],
        dropna=False,
    )[["saliency_entropy", "central_concentration_50pct"]].agg(
        ["count", "mean", "std"]
    )
else:
    feature_summary = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
feature_summary

얼굴 영역 mask가 검증되면 `saliency_concentration`으로 눈·코·입 또는
얼굴 내부/외부 비율을 별도 열로 추가할 수 있습니다. mask가 없는
상태에서 중앙 영역을 얼굴 영역이라고 부르지 않습니다.